1. Data Splitting – Stratified Split
First, we’ll split the data into 80% training and 20% testing using a stratified split so that the Failure/No Failure ratio is maintained in both datasets.
2. Distribution Validation
After splitting, we’ll check the class distribution in both train and test data to make sure the split has preserved the original distribution.
3. Handle Class Imbalance – Training Data Only
Then, we’ll handle the imbalance only in the training dataset. We can compare three approaches:
No balancing → baseline
Class Weighting → give more importance to the Failure class
SMOTE → generate synthetic samples for the minority Failure class
The test data will not be resampled or modified, so we can use it later for unbiased final evaluation.
4. Feature Scaling
After that, we’ll apply StandardScaler for the numerical features. The scaler will be fitted only on the training data and then used to transform both train and test data.

In [1]:
# libraries

import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("../data/processed/featured_cleaned_data.csv")

In [4]:
X = df.drop(columns="machine_failure")
y = df["machine_failure"]

## 80/20 Stratified Train/Test Split

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [6]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (8000, 9)
X_test: (2000, 9)
y_train: (8000,)
y_test: (2000,)


## Distribution Validation

In [7]:
print("Original Distribution:")
print(y.value_counts(normalize=True) * 100)

print("\nTraining Distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting Distribution:")
print(y_test.value_counts(normalize=True) * 100)

Original Distribution:
machine_failure
0    96.61
1     3.39
Name: proportion, dtype: float64

Training Distribution:
machine_failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64

Testing Distribution:
machine_failure
0    96.6
1     3.4
Name: proportion, dtype: float64


In [8]:
print("Original Counts:")
print(y.value_counts())

print("\nTraining Counts:")
print(y_train.value_counts())

print("\nTesting Counts:")
print(y_test.value_counts())

Original Counts:
machine_failure
0    9661
1     339
Name: count, dtype: int64

Training Counts:
machine_failure
0    7729
1     271
Name: count, dtype: int64

Testing Counts:
machine_failure
0    1932
1      68
Name: count, dtype: int64


Set Up Class Imbalance Experiments

In [9]:
print(y_train.value_counts())

print("\nPercentage:")
print(y_train.value_counts(normalize=True) * 100)

machine_failure
0    7729
1     271
Name: count, dtype: int64

Percentage:
machine_failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64


Experiment 1 → No Balancing (Baseline)

Experiment 2 → Class Weighting

Experiment 3 → SMOTE

In [11]:
X_train.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

The actual SMOTE implementation should preferably be done later inside the model pipeline, so that when we do cross-validation, SMOTE is applied only to each training fold and doesn't leak information into validation folds.

# Feature Scaling

In [12]:
numerical_columns = [
    "air_temperature_k",
    "process_temperature_k",
    "rotational_speed_rpm",
    "torque_nm",
    "tool_wear_min",
    "temperature_difference_k",
    "power_w",
    "tool_wear_torque"
]

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [14]:
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)

X_test_scaled[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

In [17]:
X_train_scaled.to_csv(
    "../data/processed/split_data/X_train_scaled.csv",
    index=False
)

X_test_scaled.to_csv(
    "../data/processed/split_data/X_test_scaled.csv",
    index=False
)

## this thing we will do while training  the model ! 

In [19]:
"""                        TRAINING DATA
                               │
              Original X_train_scaled + y_train
                               │
        ┌──────────────────────┼──────────────────────┐
        │                      │                      │
        ▼                      ▼                      ▼
 Experiment 1             Experiment 2             Experiment 3
 No Balancing             Class Weighting           SMOTE
        │                      │                      │
 Original Data          Same Original Data       SMOTE Applied
        │                      │                      │
        ▼                      ▼                      ▼
     Train Model          Train Model             Train Model
        │                      │                      │
        └──────────────────────┼──────────────────────┘
                               │
                               ▼
                    Evaluate on SAME untouched
                     X_test_scaled + y_test """

'                        TRAINING DATA\n                               │\n              Original X_train_scaled + y_train\n                               │\n        ┌──────────────────────┼──────────────────────┐\n        │                      │                      │\n        ▼                      ▼                      ▼\n Experiment 1             Experiment 2             Experiment 3\n No Balancing             Class Weighting           SMOTE\n        │                      │                      │\n Original Data          Same Original Data       SMOTE Applied\n        │                      │                      │\n        ▼                      ▼                      ▼\n     Train Model          Train Model             Train Model\n        │                      │                      │\n        └──────────────────────┼──────────────────────┘\n                               │\n                               ▼\n                    Evaluate on SAME untouched\n                    